# Known partial overlap and cropped windows
The nine-case experimental workflow is explicitly selected with `technology="test"`. Supply the actual assay separately. Ground truth is used only after alignment.


## Data and local paths
Data are not uploaded to GitHub. Download sources and preparation instructions: [DATA.md](DATA.md).
Default prepared-input folder: `data/revision/known_overlap/` in the repository; set `RAFTUP_TUTORIAL_DATA` to override it. Relative paths inside `pair.json` resolve against that file. Outputs go to `outputs/revision/03_known_overlap/`; set `RAFTUP_OUTPUT_ROOT` to override. The real-data notebooks do not substitute synthetic data when inputs are missing.


In [ ]:
from pathlib import Path
import os, json
import numpy as np
import raftup
from raftup import run_automated_alignment, run_manual_alignment
from raftup.experiments import load_pair, save_alignment, coverage
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file()), Path.cwd())
DATA = Path(os.environ.get('RAFTUP_TUTORIAL_DATA', str(REPO / 'data/revision/known_overlap'))).expanduser().resolve()
OUT = Path(os.environ.get('RAFTUP_OUTPUT_ROOT', str(REPO / 'outputs/revision/03_known_overlap'))).expanduser().resolve()
print('Inputs:', DATA, '\nOutputs:', OUT)
print('RAFT-UP:', raftup.__version__, raftup.__file__)


In [ ]:
# Set DATA or RAFTUP_TUTORIAL_DATA to your prepared input directory.
# pair.json contains file paths, exact cost-matrix cell order and spatial calibration.
a, b, cost, cfg = load_pair(DATA / 'pair.json')
print(a.shape, b.shape, cost.shape)


In [ ]:
P, decision = run_automated_alignment(a, b, cost, technology='test',
    data_technology=cfg['data_technology'], pixels_per_100um=cfg['pixels_per_100um'])
print(decision['selected_workflow'], coverage(P))
save_alignment(P, decision, OUT / 'overlap' / cfg['name'])


In [ ]:
# Optional exact-correspondence evaluation. G has the same A-by-B ordering as P.
if 'ground_truth' in cfg:
    path = Path(cfg['ground_truth'])
    G = np.load(path if path.is_absolute() else DATA / path, allow_pickle=False).astype(bool)
    predicted = np.asarray(P) > 0
    assert G.shape == predicted.shape
    correct = int((G & predicted).sum())
    print({'correct/predicted': correct / predicted.sum() if predicted.any() else np.nan,
           'correct/ground_truth': correct / G.sum() if G.any() else np.nan})


For the archived paper figure, use `figures/01_known_overlap.ipynb`. The router was calibrated on nine research cases; this example does not claim independent generalization.
